# Project 2

Welcome to the second NLP1 project. The goal of the second tutorial was to get you familiar with using pre-trained embeddings and moving towards a compositional encoder for text classification.

Given the skills you have developed in the last week, you will now be applying them within the dataset and task you have chosen during Project 1. You will be applying hands-on the experiments you have already explored in the Tutorial 2, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and errors analyses, and tackling your own research questions. 


The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

# Get to work!

1. Find a word embedding model that is appropriate for your dataset. Think about the language, the kind of text you are working with. Find evidence that your choice is a good one: 
    * Can you tokenize your texts in a way coherent with the definition of a token in the embedding model?
    * Does the model's vocabulary give enough coverage to the words in your texts? etc.
2. Explore what your chosen word embedding model captures. Some things you might consider: 
    * Is it good at representing interesting similarities (re to your choice of task)? J
    * Have you found task-specific evidence that it's better than some other reasonable alternative?
    * Do you know about limitations of this model?
    * Any limitations that you find particularly troubling for your task? 
3. Experiment with the embed-and-pool approach.  
    * What design choices have meaningful impact in classification performance (eg, embedding size, architecture design, optimisation parameters), if any?
    * Is this better than what you could achieve last week? In other words, did pretrained embeddings add to your design?
4. Experiment with a recurrent encoder (GRU or LSTM). 
    * What design choices have meaningful impact in classification performance, if any?
    * Did you investigate the impact of data size, embedding parameters (trained or not, fixed or trainable, etc)?
    * Did you struggle with over-fitting?
    * How far can this approach go? Are you able to improve meaningfully on embed-and-pool?
5. We make decisions under uncertainty all the time, are you gathering enough information to substantiate your choices? Various things can lead to small differences in results, and these differences are not always consistent. Here are some things to think about:
    * How did you structure you hyperparameter search?
    * Are you keeping an eye on experimental variance?
6. Experiment with error analysis and try to discover what characteristics of input texts are predictive of performance differences across major design choices. 
    * What analysis features did you come up with?
    * Which ones did you manage to test and how did that go?
    * What would you prioritise in the future in order to improve your systems?

Be mindful that you will be using your experiments to write up your final blog, so make sure your code is clean and neat!

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

# Tips

## Pretrained embedding models

We will post a list with some options for you. 

## Features for analysis

Features that you can interpret as an aspect of _complexity_ in light of your task's goals.

- length
- presence of rare/common words, including unknown words
- syntactic features (like parts-of-speech, syntactic complexity, syntactic voice, presence of negation, etc.)
- semantic features (eg, the average similarity of the text to training documents or some curated set of training documents, presence of words in certain semantic classes, etc.)
- task-specific features (eg, 
- model-specific features (eg, predicted label, probability assigned to predicted label, number of parameters or layers or some other model descriptor)

In [1]:
import sys
from pathlib import Path

course_root = Path.cwd()
while not (course_root / "tools").exists():
    course_root = course_root.parent

sys.path.insert(0, str(course_root))   
print("course_root:", course_root)

from part1.data import GoEmotions

ds = GoEmotions(drop_multi_labels=True, merge_train_dev=True)
ds.summary

course_root: c:\Users\akcre\NLP\nlp1-student


,Total comments,Average n tokens,Median n tokens,Max tokens
train,40856.0,15.36,15.0,47.0
test,4590.0,15.26,15.0,37.0


# Tokenizer

In [33]:
import html, subprocess

TOKENIZER = str(course_root / "tools" / "tokenizer.perl")

def tokenize_all(texts):
    texts = [str(t).replace("\n", " ") for t in texts]
    out = subprocess.run(
        ["perl", TOKENIZER, "-l", "en", "-q"],
        input="\n".join(texts), capture_output=True, encoding="utf-8", text=True
    ).stdout
    lines = out.split("\n")[:len(texts)]
    return [[html.unescape(w) for w in line.split()] for line in lines]


# Dataset

In [11]:
train_tokens = tokenize_all(ds.train["text"].tolist())

print(train_tokens[:3])
print(len(train_tokens) == len(ds.train))   

[['My', 'favourite', 'food', 'is', 'anything', 'I', 'didn', "'t", 'have', 'to', 'cook', 'myself', '.'], ['Now', 'if', 'he', 'does', 'off', 'himself', ',', 'everyone', 'will', 'think', 'hes', 'having', 'a', 'laugh', 'screwing', 'with', 'people', 'instead', 'of', 'actually', 'dead'], ['WHY', 'THE', 'FUCK', 'IS', 'BAYLESS', 'ISOING']]
True


In [8]:
import json
with open(course_root / "tokens_train.json", "w") as f:
    json.dump(tokens, f)

# Embedding

In [ ]:
from gensim.models import KeyedVectors

ft = KeyedVectors.load_word2vec_format(
    "../fasttext/models/wiki-news-300d-1M.vec",  
    binary=False,
    limit=300_000,   # drop for final run
)
print(ft.vector_size, len(ft.key_to_index))

300 300000


In [20]:
import numpy as np

def vectorize(token_lists, kv):
    dim = kv.vector_size
    X = np.zeros((len(token_lists), dim), dtype=np.float32)
    n_tokens = n_oov = 0
    oov_words = set()
    for i, toks in enumerate(token_lists):
        vecs = []
        for t in toks:
            n_tokens += 1
            if t in kv.key_to_index:
                vecs.append(kv[t])
            elif t.lower() in kv.key_to_index:   # vectors are cased; fall back to lowercase
                vecs.append(kv[t.lower()])
            else:
                oov_words.add(t)    # can create heuristics for these words
                n_oov += 1
        if vecs:
            X[i] = np.mean(vecs, axis=0)         # takes avg of all word vectors in the text
    print(f"OOV rate: {n_oov / max(n_tokens, 1):.2%}")
    print(f"first ten oov words: {sorted(oov_words)[:10]} ...")
    return X

train_tokens = tokenize_all(ds.train["text"].tolist())

X_train = vectorize(train_tokens, ft)
print(type(X_train))

OOV rate: 1.23%
first ten oov words: ["'JIN", "'SSSS", "'excuse", "'famous", "'goodboy", "'lady", "'loled", "'our", "'thas", "'thuzad"] ...
<class 'numpy.ndarray'>


In [ ]:
from clslib import Tokenizer
tok = Tokenizer()
out = tok.tokens(ds.train["text"].tolist())        # list of strings
tok.build_vocab(ds.train["text"].tolist())         # once, on train only
ids = tok.ids(ds.train["text"].tolist())          # list of ints, unknown words -> 1
print(ids[:10])
train_tokens = tok.tokens_all(ds.train["text"].tolist())   # fast batch version
print(len(train_tokens))
print(train_tokens[:10])

754644
['[', '"', 'My', 'favourite', 'food', 'is', 'anything', 'I', 'didn', "'t", 'have', 'to', 'cook', 'myself', '.', '"', ',', "'", 'Now', 'if', 'he', 'does', 'off', 'himself', ',']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
